In [0]:
# Regression quality gate. Thresholds come from conf/project.yml (`evaluation:`) unless a widget overrides them.
import sys
import os
sys.path.insert(0, os.getcwd())

import mlflow
from utils import as_bool, load_project, widget_or

cfg = load_project()
ev = cfg["evaluation"]

dbutils.widgets.text("catalog", "stock_demo")
dbutils.widgets.text("max_validation_rmse", "")          # empty -> project.yml (null = disabled)
dbutils.widgets.text("min_directional_accuracy", "")
dbutils.widgets.text("require_beat_persistence", "")
dbutils.widgets.text("persistence_tolerance", "")

max_rmse_raw = widget_or(dbutils.widgets.get("max_validation_rmse"), ev["max_validation_rmse"])
max_rmse = None if max_rmse_raw is None or str(max_rmse_raw).lower() in ("none", "null") else float(max_rmse_raw)
min_dir = float(widget_or(dbutils.widgets.get("min_directional_accuracy"), ev["min_directional_accuracy"]))
beat_persistence = as_bool(widget_or(dbutils.widgets.get("require_beat_persistence"), ev["require_beat_persistence"]))
tolerance = float(widget_or(dbutils.widgets.get("persistence_tolerance"), ev.get("persistence_tolerance", 0.0)))

# Information from the training task
run_id = dbutils.jobs.taskValues.get(taskKey="train_models", key="run_id")
best_algorithm = dbutils.jobs.taskValues.get(taskKey="train_models", key="best_algorithm")
print(f"Evaluating: {best_algorithm} | run {run_id}")

run = mlflow.get_run(run_id)
print(f"Run status: {run.info.status}")
mt = run.data.metrics

val_rmse, test_rmse = float(mt["val_rmse"]), float(mt["test_rmse"])
val_base, test_base = float(mt["val_persistence_rmse"]), float(mt["test_persistence_rmse"])
test_dir = float(mt["test_directional_accuracy"])

checks = {
    "validation_rmse_below_ceiling": max_rmse is None or val_rmse <= max_rmse,
    "beats_persistence_validation": (not beat_persistence) or val_rmse <= val_base * (1.0 + tolerance),
    "beats_persistence_test": (not beat_persistence) or test_rmse <= test_base * (1.0 + tolerance),
    "directional_accuracy_ok": test_dir >= min_dir,
}
eval_passed = all(checks.values())

print({"val_rmse": val_rmse, "val_persistence_rmse": val_base,
       "test_rmse": test_rmse, "test_persistence_rmse": test_base,
       "test_directional_accuracy": test_dir})
print(f"Persistence tolerance: {tolerance:.2%} | val ratio: {val_rmse / val_base:.4f} | test ratio: {test_rmse / test_base:.4f}")
print("Checks:", checks)
print(f"Evaluation passed: {eval_passed}")

# Pass result to next task
dbutils.jobs.taskValues.set(key="eval_passed", value=bool(eval_passed))
dbutils.jobs.taskValues.set(key="run_id", value=run_id)